# フルーツキャッチ（p5.js ゲーム）

上から落ちてくるフルーツをカゴでキャッチするゲームです。

## ルール
- フルーツをカゴで受け止めると **1 点**
- 取り逃がすとライフが 1 つ減り、ライフが 0 になるとゲームオーバー
- スコアが上がるほどフルーツが速くなります

## 操作
- **マウスを左右に動かす**: カゴを移動
- **クリック**: ゲームオーバー後にもう一度遊ぶ

## このノートブックの使い方

- コードセルを上から順番に **Shift + Enter** で実行し、最後の `%show` セルを実行するとゲーム画面が表示されます。
- キーボードで操作するゲームは、**最初にゲーム画面をクリック** してから操作してください（クリックでキー入力が画面に届くようになります）。
- コードを書き換えたら、そのセルを実行し直してから `%show` をもう一度実行すると、新しいゲームになります。
- 動かなくなったら、メニューの **Kernel → Restart Kernel and Clear Outputs of All Cells...** で最初からやり直せます。

p5.js の基本は `p5-tutorial.ipynb` で学べます。

## 1. ゲームの状態を表す変数

ゲーム中に変化する値（カゴの位置、フルーツの一覧、スコア、ライフなど）は `draw()` の外で変数として持ちます。

In [ ]:
let basketX = 200;     // カゴの x 座標
let fruits = [];       // 落ちてくるフルーツの配列
let score = 0;         // スコア
let lives = 3;         // ライフ
let gameOver = false;  // ゲームオーバーかどうか

## 2. フルーツのクラス

フルーツ 1 個分のデータ（位置・大きさ・速さ・色）と動きをクラスにまとめます。
`isCaught()` はカゴに入ったか、`isMissed()` は画面の下に落ちたかを判定します。

In [ ]:
class Fruit {
  constructor() {
    this.x = random(20, width - 20);            // 横位置はランダム
    this.y = -20;                               // 画面の上の外からスタート
    this.d = random(22, 36);                    // 直径
    this.speed = random(2, 4) + score * 0.15;   // スコアが高いほど速い
    this.c = color(random(150, 255), random(60, 200), random(50, 150));
  }

  update() {
    this.y += this.speed;
  }

  show() {
    noStroke();
    fill(this.c);
    circle(this.x, this.y, this.d);
    fill(60, 160, 60);                          // 葉っぱ
    ellipse(this.x + 4, this.y - this.d / 2, 10, 5);
  }

  isCaught() {
    // カゴ（幅 80、y = height - 20）の高さに来ていて、横位置がカゴの範囲内
    return this.y > height - 40 && this.y < height - 10 && abs(this.x - basketX) < 40;
  }

  isMissed() {
    return this.y > height + 20;
  }
}

## 3. setup と draw

`draw()` では「カゴを描く → フルーツを増やす → フルーツを動かして判定 → スコア表示」の順に処理します。
配列から要素を削除しながらループするときは、**後ろから** 回すのが安全です。

In [ ]:
function setup() {
  createCanvas(400, 400);
  textFont("sans-serif");
}

function draw() {
  background(135, 206, 235);   // 空色

  if (gameOver) {
    drawGameOver();
    return;                    // ゲームオーバー中は以下を実行しない
  }

  // カゴ: マウスに追従（画面の外に出ないように constrain で制限）
  basketX = constrain(mouseX, 40, width - 40);
  fill(139, 69, 19);
  noStroke();
  rect(basketX - 40, height - 30, 80, 20, 6);

  // 40 フレームごとに新しいフルーツを追加
  if (frameCount % 40 === 0) {
    fruits.push(new Fruit());
  }

  // フルーツを動かして判定（削除するので後ろからループ）
  for (let i = fruits.length - 1; i >= 0; i--) {
    const f = fruits[i];
    f.update();
    f.show();
    if (f.isCaught()) {
      score++;
      fruits.splice(i, 1);           // 配列から取り除く
    } else if (f.isMissed()) {
      lives--;
      fruits.splice(i, 1);
      if (lives <= 0) {
        gameOver = true;
      }
    }
  }

  drawHUD();
}

// スコアとライフの表示
function drawHUD() {
  fill(0);
  textSize(18);
  textAlign(LEFT, TOP);
  text("スコア: " + score, 10, 10);
  textAlign(RIGHT, TOP);
  text("ライフ: " + "♥".repeat(lives), width - 10, 10);
}

// ゲームオーバー画面
function drawGameOver() {
  fill(0, 160);
  rect(0, 0, width, height);
  fill(255);
  textAlign(CENTER, CENTER);
  textSize(36);
  text("ゲームオーバー", width / 2, height / 2 - 30);
  textSize(20);
  text("スコア: " + score, width / 2, height / 2 + 15);
  textSize(16);
  text("クリックでもう一度", width / 2, height / 2 + 50);
}

## 4. 入力とリセット

In [ ]:
function mousePressed() {
  if (gameOver) {
    resetGame();
  }
}

function resetGame() {
  fruits = [];
  score = 0;
  lives = 3;
  gameOver = false;
}

In [ ]:
%show 100% 410px

## 改造のヒント

- `frameCount % 40` の `40` を小さくすると、フルーツがたくさん降ってきます
- 「腐ったフルーツ（取るとライフが減る）」を追加してみましょう（`Fruit` に種類のプロパティを持たせる）
- `keyPressed()` で左右キーでもカゴを動かせるようにしてみましょう
- ハイスコアを変数に保存して表示してみましょう